# Feature Engineering Tracking with MLflow

## Topic Goal


Feature engineering changes can significantly impact model performance.  
In production MLOps, we should track:

1. Original feature list
2. Engineered feature list
3. Feature engineering version
4. Transformation logic
5. Baseline model performance
6. Engineered model performance
7. Feature metadata artifacts
8. Feature importance artifacts
9. Same-run model logging with signature and input example
10. Model registration using the same-run `model_uri`

This notebook compares a **baseline model** with an **engineered-feature model** and logs the engineered model properly.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, matplotlib, and JSON utilities.

The notebook will generate feature engineering metadata and model comparison artifacts.


In [ ]:
# Import os to create folders and manage local paths.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import json to save feature engineering metadata.
import json

# Import datetime to record metadata creation time.
from datetime import datetime

# Import MLflow for tracking, model logging, and registry.
import mlflow

# Import MLflow scikit-learn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input/output schema.
from mlflow.models.signature import infer_signature

# Import pandas for data processing.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import matplotlib for feature importance plots.
import matplotlib.pyplot as plt

# Import train_test_split for dataset splitting.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for preprocessing different feature groups.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical features.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical features.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Local Folders

This notebook uses local MLflow tracking.

All generated files are saved under `outputs/`.


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_14_feature_engineering_tracking"

# Define run name.
RUN_NAME = "14_feature_engineering_tracking_same_run_usecase"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for artifacts.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for supporting files.
os.makedirs("artifacts", exist_ok=True)

# Configure local file-based MLflow tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Define feature engineering version.
FEATURE_ENGINEERING_VERSION = "v2_business_features"

# Print setup information.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Feature engineering version:", FEATURE_ENGINEERING_VERSION)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)


## 3. Load Dataset

We load the customer churn dataset.

The target column is:

```text
churn
```


In [ ]:
# Check whether dataset exists.
if not os.path.exists(DATA_PATH):
    # Raise a clear error if dataset is missing.
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load dataset.
df = pd.read_csv(DATA_PATH)

# Display first five records.
display(df.head())

# Define target column.
target_column = "churn"

# Define ID column.
id_column = "customer_id"

# Print dataset shape.
print("Dataset shape:", df.shape)


## 4. Baseline Feature Preparation

First, we train a baseline model using the original features.

This gives us a reference point before feature engineering.


In [ ]:
# Create baseline feature matrix by dropping ID and target.
X_baseline = df.drop(columns=[id_column, target_column])

# Create target vector.
y = df[target_column]

# Store original feature list.
original_features = X_baseline.columns.tolist()

# Identify baseline categorical columns.
baseline_categorical_columns = X_baseline.select_dtypes(include=["object"]).columns.tolist()

# Identify baseline numerical columns.
baseline_numerical_columns = X_baseline.select_dtypes(exclude=["object"]).columns.tolist()

# Create baseline preprocessor.
baseline_preprocessor = ColumnTransformer(
    transformers=[
        # Scale original numerical columns.
        ("num", StandardScaler(), baseline_numerical_columns),

        # One-hot encode original categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), baseline_categorical_columns),
    ]
)

# Split baseline data.
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_baseline,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print original features.
print("Original features:", original_features)
print("Baseline categorical columns:", baseline_categorical_columns)
print("Baseline numerical columns:", baseline_numerical_columns)


## 5. Train Baseline Model

This baseline run is logged separately for comparison only.

The final registered model will be the engineered-feature model.


In [ ]:
# Create baseline model.
baseline_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create baseline pipeline.
baseline_pipeline = Pipeline(steps=[
    # Apply baseline preprocessing.
    ("preprocessor", baseline_preprocessor),

    # Train baseline model.
    ("model", baseline_model),
])

# Train baseline pipeline.
baseline_pipeline.fit(X_train_base, y_train)

# Predict using baseline model.
baseline_predictions = baseline_pipeline.predict(X_test_base)

# Calculate baseline metrics.
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_precision = precision_score(y_test, baseline_predictions, zero_division=0)
baseline_recall = recall_score(y_test, baseline_predictions, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_predictions, zero_division=0)

# Start a comparison-only baseline MLflow run.
with mlflow.start_run(run_name="baseline_original_features_comparison"):

    # Log run type.
    mlflow.log_param("run_type", "baseline_comparison")

    # Log feature engineering version.
    mlflow.log_param("feature_engineering_version", "none_baseline")

    # Log original feature count.
    mlflow.log_param("feature_count", len(original_features))

    # Log baseline metrics.
    mlflow.log_metric("accuracy", baseline_accuracy)
    mlflow.log_metric("precision", baseline_precision)
    mlflow.log_metric("recall", baseline_recall)
    mlflow.log_metric("f1_score", baseline_f1)

# Print baseline results.
print("Baseline accuracy:", baseline_accuracy)
print("Baseline precision:", baseline_precision)
print("Baseline recall:", baseline_recall)
print("Baseline F1-score:", baseline_f1)


## 6. Create Engineered Features

Now we create multiple business-driven engineered features.

These features are examples of how domain knowledge can improve model performance:

| Engineered Feature | Meaning |
|---|---|
| `support_tickets_per_tenure` | Support pressure normalized by customer tenure |
| `monthly_charge_per_gb` | Price paid per GB of usage |
| `is_high_support_customer` | Customer has more support tickets than median |
| `is_long_tenure_customer` | Customer has tenure greater than or equal to 36 months |
| `is_high_value_customer` | Customer has high monthly charges |
| `contract_risk_score` | Numeric risk score based on contract type |
| `plan_risk_score` | Numeric risk score based on plan type |


In [ ]:
# Create a copy of original dataframe for feature engineering.
df_fe = df.copy()

# Create support pressure feature.
df_fe["support_tickets_per_tenure"] = df_fe["support_tickets"] / (df_fe["tenure_months"] + 1)

# Create price-per-usage feature.
df_fe["monthly_charge_per_gb"] = df_fe["monthly_charges"] / (df_fe["data_usage_gb"] + 1)

# Create high support flag based on median support tickets.
df_fe["is_high_support_customer"] = (df_fe["support_tickets"] > df_fe["support_tickets"].median()).astype(int)

# Create long tenure flag.
df_fe["is_long_tenure_customer"] = (df_fe["tenure_months"] >= 36).astype(int)

# Create high value customer flag based on median monthly charge.
df_fe["is_high_value_customer"] = (df_fe["monthly_charges"] > df_fe["monthly_charges"].median()).astype(int)

# Create contract risk mapping.
contract_risk_map = {
    "Month-to-month": 3,
    "One year": 2,
    "Two year": 1,
}

# Create contract risk score.
df_fe["contract_risk_score"] = df_fe["contract_type"].map(contract_risk_map).fillna(2)

# Create plan risk mapping.
plan_risk_map = {
    "Basic": 3,
    "Standard": 2,
    "Premium": 1,
    "Enterprise": 1,
}

# Create plan risk score.
df_fe["plan_risk_score"] = df_fe["plan_type"].map(plan_risk_map).fillna(2)

# Define engineered feature list.
engineered_features = [
    "support_tickets_per_tenure",
    "monthly_charge_per_gb",
    "is_high_support_customer",
    "is_long_tenure_customer",
    "is_high_value_customer",
    "contract_risk_score",
    "plan_risk_score",
]

# Create engineered feature matrix.
X_engineered = df_fe.drop(columns=[id_column, target_column])

# Create complete engineered feature list.
all_engineered_feature_columns = X_engineered.columns.tolist()

# Print engineered features.
print("Engineered features:", engineered_features)
print("Total feature count after engineering:", len(all_engineered_feature_columns))

# Display engineered feature sample.
display(df_fe[[id_column] + engineered_features + [target_column]].head())


## 7. Save Feature Engineering Metadata

We save feature engineering metadata as artifacts.

This helps future users understand:

- what features were added
- why they were added
- what transformation logic was used
- which version of feature engineering was applied


In [ ]:
# Create feature engineering metadata.
feature_engineering_metadata = {
    "metadata_created_at": datetime.now().isoformat(),
    "feature_engineering_version": FEATURE_ENGINEERING_VERSION,
    "original_features": original_features,
    "engineered_features": engineered_features,
    "final_feature_columns": all_engineered_feature_columns,
    "original_feature_count": len(original_features),
    "engineered_feature_count": len(engineered_features),
    "final_feature_count": len(all_engineered_feature_columns),
    "transformation_logic": {
        "support_tickets_per_tenure": "support_tickets / (tenure_months + 1)",
        "monthly_charge_per_gb": "monthly_charges / (data_usage_gb + 1)",
        "is_high_support_customer": "support_tickets > median(support_tickets)",
        "is_long_tenure_customer": "tenure_months >= 36",
        "is_high_value_customer": "monthly_charges > median(monthly_charges)",
        "contract_risk_score": "Month-to-month=3, One year=2, Two year=1",
        "plan_risk_score": "Basic=3, Standard=2, Premium=1, Enterprise=1",
    },
}

# Save metadata JSON.
with open("outputs/feature_engineering_metadata.json", "w") as file:
    json.dump(feature_engineering_metadata, file, indent=4)

# Save engineered feature list.
with open("outputs/engineered_features.txt", "w") as file:
    for feature in engineered_features:
        file.write(feature + "\n")

# Save final feature list.
with open("outputs/final_feature_columns.txt", "w") as file:
    for feature in all_engineered_feature_columns:
        file.write(feature + "\n")

# Print artifact names.
print("Saved feature engineering artifacts:")
print("- outputs/feature_engineering_metadata.json")
print("- outputs/engineered_features.txt")
print("- outputs/final_feature_columns.txt")


## 8. Train Engineered-Feature Model

Now we train the model using the engineered feature set.


In [ ]:
# Identify categorical columns after feature engineering.
engineered_categorical_columns = X_engineered.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns after feature engineering.
engineered_numerical_columns = X_engineered.select_dtypes(exclude=["object"]).columns.tolist()

# Create engineered preprocessor.
engineered_preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical engineered columns.
        ("num", StandardScaler(), engineered_numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), engineered_categorical_columns),
    ]
)

# Split engineered data.
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    X_engineered,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Create engineered model.
engineered_model = RandomForestClassifier(
    n_estimators=180,
    max_depth=8,
    random_state=42
)

# Create engineered pipeline.
engineered_pipeline = Pipeline(steps=[
    # Apply engineered preprocessing.
    ("preprocessor", engineered_preprocessor),

    # Train engineered model.
    ("model", engineered_model),
])

# Train engineered pipeline.
engineered_pipeline.fit(X_train_eng, y_train_eng)

# Predict using engineered model.
engineered_predictions = engineered_pipeline.predict(X_test_eng)

# Calculate engineered metrics.
engineered_accuracy = accuracy_score(y_test_eng, engineered_predictions)
engineered_precision = precision_score(y_test_eng, engineered_predictions, zero_division=0)
engineered_recall = recall_score(y_test_eng, engineered_predictions, zero_division=0)
engineered_f1 = f1_score(y_test_eng, engineered_predictions, zero_division=0)

# Create classification report.
engineered_report = classification_report(y_test_eng, engineered_predictions, zero_division=0)

# Print engineered metrics.
print("Engineered accuracy:", engineered_accuracy)
print("Engineered precision:", engineered_precision)
print("Engineered recall:", engineered_recall)
print("Engineered F1-score:", engineered_f1)
print(engineered_report)


## 9. Baseline vs Engineered Model Comparison

We compare baseline model performance against engineered-feature model performance.

This is the core idea of feature engineering tracking.


In [ ]:
# Create model comparison dataframe.
comparison_df = pd.DataFrame([
    {
        "model_version": "baseline_original_features",
        "feature_engineering_version": "none_baseline",
        "feature_count": len(original_features),
        "accuracy": baseline_accuracy,
        "precision": baseline_precision,
        "recall": baseline_recall,
        "f1_score": baseline_f1,
    },
    {
        "model_version": "engineered_features",
        "feature_engineering_version": FEATURE_ENGINEERING_VERSION,
        "feature_count": len(all_engineered_feature_columns),
        "accuracy": engineered_accuracy,
        "precision": engineered_precision,
        "recall": engineered_recall,
        "f1_score": engineered_f1,
    },
])

# Calculate F1 improvement.
f1_improvement = engineered_f1 - baseline_f1

# Save comparison artifact.
comparison_df.to_csv("outputs/baseline_vs_engineered_comparison.csv", index=False)

# Display comparison.
display(comparison_df)

# Print improvement.
print("F1 improvement:", f1_improvement)


## 10. Create Feature Importance Artifact

For tree-based models, feature importance helps explain which features contributed most.

Because preprocessing expands categorical columns, we extract transformed feature names from the fitted pipeline.


In [ ]:
# Get transformed feature names from preprocessor.
transformed_feature_names = engineered_pipeline.named_steps["preprocessor"].get_feature_names_out()

# Get feature importance values from Random Forest.
feature_importances = engineered_pipeline.named_steps["model"].feature_importances_

# Create feature importance dataframe.
feature_importance_df = pd.DataFrame({
    "feature": transformed_feature_names,
    "importance": feature_importances,
}).sort_values("importance", ascending=False)

# Save feature importance CSV.
feature_importance_df.to_csv("outputs/feature_importance.csv", index=False)

# Select top 15 features.
top_features = feature_importance_df.head(15)

# Plot top feature importances.
plt.figure(figsize=(10, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("outputs/feature_importance.png")
plt.close()

# Display top features.
display(top_features)


## 11. Infer Signature for Engineered Model

The registered model should use the engineered feature input schema.

So we infer the signature using `X_test_eng`, not the baseline feature matrix.


In [ ]:
# Create input example from engineered test data.
input_example = X_test_eng.head(5)

# Generate sample predictions for signature.
sample_predictions = engineered_pipeline.predict(input_example)

# Infer model signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 12. Same-Run MLflow Logging and Registry

This is the main production-style run for the engineered-feature model.

Inside the **same MLflow run**, we log:

- feature engineering version
- original features
- engineered features
- final feature count
- baseline metrics
- engineered model metrics
- improvement metrics
- feature metadata artifacts
- feature importance artifacts
- model with signature and input example
- model URI

Then we register the model using the same `model_uri`.


In [ ]:
# Start the SAME MLflow run for engineered model, signature, feature metadata, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log feature engineering version.
    mlflow.log_param("feature_engineering_version", FEATURE_ENGINEERING_VERSION)

    # Log original feature count.
    mlflow.log_param("original_feature_count", len(original_features))

    # Log engineered feature count.
    mlflow.log_param("engineered_feature_count", len(engineered_features))

    # Log final feature count.
    mlflow.log_param("final_feature_count", len(all_engineered_feature_columns))

    # Log engineered feature names.
    mlflow.log_param("engineered_features", ",".join(engineered_features))

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Set feature engineering version as a searchable tag.
    mlflow.set_tag("feature_engineering_version", FEATURE_ENGINEERING_VERSION)

    # Set run purpose.
    mlflow.set_tag("run_purpose", "feature_engineering_tracking")

    # Log baseline metrics.
    mlflow.log_metric("baseline_accuracy", baseline_accuracy)
    mlflow.log_metric("baseline_precision", baseline_precision)
    mlflow.log_metric("baseline_recall", baseline_recall)
    mlflow.log_metric("baseline_f1_score", baseline_f1)

    # Log engineered model metrics.
    mlflow.log_metric("engineered_accuracy", engineered_accuracy)
    mlflow.log_metric("engineered_precision", engineered_precision)
    mlflow.log_metric("engineered_recall", engineered_recall)
    mlflow.log_metric("engineered_f1_score", engineered_f1)

    # Log improvement.
    mlflow.log_metric("f1_improvement", f1_improvement)

    # Log standard metric names for easier MLflow comparison.
    mlflow.log_metric("accuracy", engineered_accuracy)
    mlflow.log_metric("precision", engineered_precision)
    mlflow.log_metric("recall", engineered_recall)
    mlflow.log_metric("f1_score", engineered_f1)

    # Save engineered classification report.
    with open("outputs/engineered_classification_report.txt", "w") as file:
        file.write(engineered_report)

    # Log feature engineering artifacts.
    mlflow.log_artifact("outputs/feature_engineering_metadata.json")
    mlflow.log_artifact("outputs/engineered_features.txt")
    mlflow.log_artifact("outputs/final_feature_columns.txt")
    mlflow.log_artifact("outputs/baseline_vs_engineered_comparison.csv")
    mlflow.log_artifact("outputs/feature_importance.csv")
    mlflow.log_artifact("outputs/feature_importance.png")
    mlflow.log_artifact("outputs/engineered_classification_report.txt")

    # Log engineered model with signature and input example in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=engineered_pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register model using the same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print model and registry details.
print("Same-run model URI:", model_uri)
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)


## 13. Verify Artifacts

This section verifies that feature engineering files were created.


In [ ]:
# List important generated artifacts.
expected_artifacts = [
    "outputs/feature_engineering_metadata.json",
    "outputs/engineered_features.txt",
    "outputs/final_feature_columns.txt",
    "outputs/baseline_vs_engineered_comparison.csv",
    "outputs/feature_importance.csv",
    "outputs/feature_importance.png",
    "outputs/engineered_classification_report.txt",
]

# Check each artifact.
for artifact_path in expected_artifacts:
    # Print whether artifact exists.
    print(artifact_path, "->", os.path.exists(artifact_path))




- Feature engineering should be treated as part of the model lifecycle.
- If feature logic changes, the model input behavior changes.
- MLflow should track the feature engineering version.
- Always log original features, engineered features, and final feature columns.
- Compare baseline performance with engineered-feature performance.
- Feature importance artifacts help explain whether engineered features are useful.
- The final registered model should include preprocessing and engineered features in one pipeline.
- The model signature should be inferred from engineered features, not the original baseline feature matrix.
